# Run MODFLOW-96 with FloPy

Local FloPy workflow (no Tapis calls):
1. Load an existing model from shared model storage
2. Point it to a local tutorial output workspace
3. Write and run model input
4. Check expected output files


In [ ]:
from pathlib import Path
from shutil import which
import json
import os
import re
import zipfile
from urllib.parse import parse_qs, quote, unquote, urlparse
from urllib.request import Request, urlopen
import flopy


In [ ]:
# Paths and executable configuration
local_modeldir = Path(r"/corral-repl/tacc/aci/PT2050/projects/PTDATAX-272/workingGAMs/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate")
modeldir = None  # set below based on USE_CKAN_DATA
run_dir = Path(r"model_output_directory/modflow_96")
exe_name = r"mf96"

# Optional CKAN dataset download
# Set USE_CKAN_DATA=True and provide CKAN_DATASET_URL to stage resources into ./data
USE_CKAN_DATA = False
CKAN_DATASET_URL = "https://ckan.tacc.utexas.edu/dataset/REPLACE_WITH_DATASET_SLUG"
CKAN_DATA_ROOT = Path("./data")
CKAN_MODEL_SUBDIR = ""  # Optional path under downloaded dataset directory
CKAN_EXTRACT_ZIPS = True
CKAN_OVERWRITE = False
CKAN_MAX_RESOURCES = None  # Optional int limit for first N resources


def _safe_filename(value):
    clean = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("._")
    return clean or "resource"


def _dataset_base_and_slug(dataset_url):
    parsed = urlparse(dataset_url)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError(f"Invalid CKAN dataset URL: {dataset_url}")

    parts = [p for p in parsed.path.strip("/").split("/") if p]
    slug = ""
    if "dataset" in parts:
        i = parts.index("dataset")
        if i + 1 < len(parts):
            slug = parts[i + 1]
    if not slug:
        q = parse_qs(parsed.query)
        slug = q.get("id", [""])[0]
    if not slug and parts:
        slug = parts[-1]
    if not slug:
        raise ValueError(f"Could not determine dataset slug from URL: {dataset_url}")

    return f"{parsed.scheme}://{parsed.netloc}", unquote(slug)


def _open_url(url):
    req = Request(url, headers={"User-Agent": "flopy-interactive-notebook/1.0"})
    return urlopen(req, timeout=120)


def _load_package(dataset_url):
    base_url, dataset_slug = _dataset_base_and_slug(dataset_url)
    api_url = f"{base_url}/api/3/action/package_show?id={quote(dataset_slug)}"
    with _open_url(api_url) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    if not payload.get("success"):
        raise RuntimeError(f"CKAN API package_show failed for '{dataset_slug}': {payload}")
    return base_url, dataset_slug, payload["result"]


def _download_to(url, target_path):
    target_path.parent.mkdir(parents=True, exist_ok=True)
    with _open_url(url) as resp, open(target_path, "wb") as dst:
        dst.write(resp.read())


def download_ckan_dataset(dataset_url, data_root, extract_zips=True, overwrite=False, max_resources=None):
    base_url, dataset_slug, package = _load_package(dataset_url)
    dataset_dir = Path(data_root) / _safe_filename(dataset_slug)
    resources_dir = dataset_dir / "resources"
    resources_dir.mkdir(parents=True, exist_ok=True)

    resources = package.get("resources", [])
    if max_resources is not None:
        resources = resources[: int(max_resources)]

    print(f"CKAN base URL: {base_url}")
    print(f"Dataset slug: {dataset_slug}")
    print(f"Resource count: {len(resources)}")

    for i, res in enumerate(resources, start=1):
        url = res.get("url")
        if not url:
            print(f"  - Skipping resource {i} (missing url)")
            continue

        url_path_name = Path(urlparse(url).path).name
        raw_name = res.get("name") or url_path_name or f"resource_{i}"
        fname = _safe_filename(raw_name)

        if "." not in fname and "." in url_path_name:
            fname = f"{fname}{Path(url_path_name).suffix}"

        resource_id = (res.get("id") or f"r{i}").replace("-", "")[:8]
        out_path = resources_dir / f"{i:03d}_{resource_id}_{fname}"

        if out_path.exists() and not overwrite:
            print(f"  - Exists, skipping: {out_path}")
        else:
            print(f"  - Downloading {url} -> {out_path}")
            _download_to(url, out_path)

        if extract_zips and out_path.suffix.lower() == ".zip":
            extract_dir = dataset_dir / "extracted" / out_path.stem
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(out_path, "r") as zf:
                zf.extractall(extract_dir)
            print(f"    Extracted zip -> {extract_dir}")

    return dataset_dir


if USE_CKAN_DATA:
    if "REPLACE_WITH_DATASET_SLUG" in CKAN_DATASET_URL:
        raise ValueError("Set CKAN_DATASET_URL to a real dataset page URL before USE_CKAN_DATA=True")

    staged_dataset_dir = download_ckan_dataset(
        CKAN_DATASET_URL,
        CKAN_DATA_ROOT,
        extract_zips=CKAN_EXTRACT_ZIPS,
        overwrite=CKAN_OVERWRITE,
        max_resources=CKAN_MAX_RESOURCES,
    )

    modeldir = staged_dataset_dir / CKAN_MODEL_SUBDIR if CKAN_MODEL_SUBDIR else staged_dataset_dir
    print(f"Using CKAN-staged modeldir: {modeldir}")
else:
    modeldir = local_modeldir
    print(f"Using local modeldir: {modeldir}")

# Auto-install executable with FloPy utility if missing on PATH
if which(exe_name) is None:
    bindir = Path("/tmp/bin")
    bindir.mkdir(parents=True, exist_ok=True)
    os.environ["PATH"] = f"{bindir}{os.pathsep}{os.environ.get('PATH', '')}"

    print(f"Attempting to install '{exe_name}' into {bindir} using flopy.utils.get_modflow...")
    try:
        import inspect

        gm_kwargs = {}
        gm_sig = inspect.signature(flopy.utils.get_modflow)
        if "subset" in gm_sig.parameters:
            gm_kwargs["subset"] = [exe_name]

        flopy.utils.get_modflow(str(bindir), **gm_kwargs)
    except Exception as err:
        print(f"Auto-install attempt with subset failed: {err}")
        print("Retrying full executable bundle install...")
        try:
            flopy.utils.get_modflow(str(bindir))
        except Exception as err2:
            print(f"Auto-install failed: {err2}")

print(f"FloPy version: {flopy.__version__}")
print(f"CKAN dataset URL (optional): {CKAN_DATASET_URL}")
print(f"Source model directory: {modeldir}")
print(f"Run workspace: {run_dir}")
print(f"Executable: {exe_name}")

if not modeldir.exists():
    raise FileNotFoundError(f"Model directory not found: {modeldir}")

exe_path = which(exe_name)
if exe_path is None:
    print(f"WARNING: executable '{exe_name}' is not on PATH.")
    print("Set exe_name to a full path if needed before running model.")
else:
    print(f"Found executable: {exe_path}")

run_dir.mkdir(parents=True, exist_ok=True)



In [ ]:
preferred_namefiles = ["trnt_h_ss.nam", "model.nam"]
namefile = next((n for n in preferred_namefiles if (modeldir / n).exists()), None)
if namefile is None:
    nam_candidates = sorted(modeldir.glob("*.nam"))
    if not nam_candidates:
        raise FileNotFoundError(f"No MODFLOW-96 .nam file found in {modeldir}")
    namefile = nam_candidates[0].name

print(f"Using name file: {namefile}")

model_obj = flopy.modflow.Modflow.load(
    f=namefile,
    version="mf2k",
    exe_name=exe_name,
    model_ws=str(modeldir),
    check=False,
    verbose=True,
)
model_obj.change_model_ws(str(run_dir), reset_external=True)


In [ ]:
# Write model files into local run workspace
write_method = getattr(model_obj, "write_simulation", None)
if callable(write_method):
    write_method()
else:
    model_obj.write_input()


In [ ]:
# Run model through FloPy
run_method = getattr(model_obj, "run_simulation", None)
if callable(run_method):
    success, buff = run_method()
else:
    success, buff = model_obj.run_model(silent=False, report=True)

print(f"Success: {success}")
if not success:
    print("Model did not terminate normally.")


In [ ]:
# Output checks
list_candidates = sorted(run_dir.glob("*.lst"))
for f in list_candidates[:5]:
    print(f"List file: {f.name}")

if not list_candidates:
    print("No .lst file found yet in run workspace.")
